In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
import pickle as pkl
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
from src.plotting_tools.cms_format import cms_format_fig, cms_style
cms_style()
from src.assets.output_dir import output_dir
output_dir
from src.plotting_tools.Bins import Bins, bins
from src.plotting_tools.SysHist import SysHist
from src.data_tools.StackPlotter import get_stack_plotter

In [ ]:
era = "2017"
ismc = 0
xrange = (120,401)

In [ ]:
outname="{}/abcd/abcd_dict_data_{}_ismc{}_v2.pkl".format(output_dir, era, ismc)
with open(outname,'rb') as f:
    abcd = pkl.load(f)

In [ ]:
##
## create fit variations
##

In [ ]:
CR13_hist = SysHist.from_dict(abcd['CR13'])
hist = CR13_hist.reduce_range(*xrange)
hist_rebin = hist.rebin(bins.bin_edges).reduce_range(*xrange)
fig, ax = plt.subplots()
hist_density = hist.make_density_hist()
hist_density.draw(ax)

In [ ]:
from src.general.functions import make_bpoly, lognorm, log_norm_np, log_norm_unp
from scipy.optimize import curve_fit
import copy 
from src.plotting_tools.utils import rebin_np


In [ ]:
def make_variations(_nBin, array):
    x = array[_nBin]
    std_dev = x**.5
    up, down = copy.copy(array), copy.copy(array)
    up[_nBin]+=std_dev
    down[_nBin]-=std_dev
    return up, down
    

In [ ]:
def fit(_x, _y):
    total_events = _y.sum()
    popt, pcov = curve_fit(ln_function, _x,_y,
                       p0=[total_events*3, .8, 80, 70], bounds = ([0, .2, 50, 50], [total_events*100, 1, 100, 100])
                          )
    return popt, pcov

def lognorm_variable_width(*args, widths=1):
    y = log_norm_np(*args)
    return y*widths

ln_function = lambda *x: lognorm_variable_width(*x, widths=hist.calc_bin_widths())    
   

In [ ]:
x = hist_density.calc_bin_centers()
xp = hist_rebin.calc_bin_centers()
bin_widths = hist_rebin.calc_bin_widths()
ratios_CR13 = {}
for i in range(len(hist_density.nominal)):
    up, down = make_variations(i, hist.nominal)
    #make fits
    poptup, pcov = fit(x, up)                                                               
    poptdown, pcov = fit(x, down)                                                               
    poptnom, pcov = fit(x, hist.nominal) 
    
    up_fit = ln_function(x, *poptup)
    down_fit = ln_function(x, *poptdown)
    nom_fit = ln_function(x, *poptnom)
    
    up_fit = rebin_np(x, hist_rebin.bin_edges, up_fit) 
    down_fit = rebin_np(x, hist_rebin.bin_edges, down_fit) 
    nom_fit = rebin_np(x, hist_rebin.bin_edges, nom_fit) 

    
    #make_plot
    if (i % 10)==0:
        print(i)
        fig, axs = plt.subplots(2,1)
        top, bottom = axs
        
        top.plot(x, up, label='Up', color='blue', ls=":")
        top.plot(xp, up_fit/bin_widths, label='Up Fit', color='blue')
        
        top.plot(x, down, label='Down', color='red', ls=":")
        top.plot(xp, down_fit/bin_widths, label='Down Fit', color='red')
        
        top.plot(x, hist.nominal, color='black', ls=":")
        top.plot(xp, nom_fit/bin_widths, label='Nom Fit', color='black')
        
        bottom.plot(xp, up_fit/nom_fit, color='blue')
        bottom.plot(xp, nom_fit/nom_fit, color='black')
        bottom.plot(xp, down_fit/nom_fit, color='red')
        bottom.set_ylim(.9,1.1)
        
        top.legend(title='Bin: {}'.format(i))
        top.set_xlabel('$m_{\ell\ell}$ [GeV]')
        top.set_ylabel('Counts')
        cms_format_fig(era, top, "\emph{Preliminary}")
        
        fig.savefig(f'{output_dir}/abcd_fit_variations/CR13_{era}_{i}.png')        
    ratios_CR13[i] = {'up': up_fit/nom_fit, "down": down_fit/nom_fit}

In [ ]:
CR23_hist = SysHist.from_dict(abcd['CR23'])
hist = CR23_hist.reduce_range(*xrange)
hist_rebin = hist.rebin(bins.bin_edges).reduce_range(*xrange)
fig, ax = plt.subplots()
hist_density = hist.make_density_hist()
hist_density.draw(ax)

In [ ]:
x = hist_density.calc_bin_centers()
xp = hist_rebin.calc_bin_centers()
bin_widths = hist_rebin.calc_bin_widths()
ratios_CR23 = {}
for i in range(len(hist_density.nominal)):
    up, down = make_variations(i, hist.nominal)
    #make fits
    poptup, pcov = fit(x, up)                                                               
    poptdown, pcov = fit(x, down)                                                               
    poptnom, pcov = fit(x, hist.nominal) 
    
    up_fit = ln_function(x, *poptup)
    down_fit = ln_function(x, *poptdown)
    nom_fit = ln_function(x, *poptnom)
    up_fit = rebin_np(x, hist_rebin.bin_edges, up_fit) 
    down_fit = rebin_np(x, hist_rebin.bin_edges, down_fit) 
    nom_fit = rebin_np(x, hist_rebin.bin_edges, nom_fit) 
    
    #make_plot
    if (i % 10)==0:
        print(i)
        fig, axs = plt.subplots(2,1)
        top, bottom = axs
        
        top.plot(x, up, label='Up', color='blue', ls=":")
        top.plot(xp, up_fit/bin_widths, label='Up Fit', color='blue')
        
        top.plot(x, down, label='Down', color='red', ls=":")
        top.plot(xp, down_fit/bin_widths, label='Down Fit', color='red')
        
        top.plot(x, hist.nominal, color='black', ls=":")
        top.plot(xp, nom_fit/bin_widths, label='Nom Fit', color='black')
        
        bottom.plot(xp, up_fit/nom_fit, color='blue')
        bottom.plot(xp, nom_fit/nom_fit, color='black')
        bottom.plot(xp, down_fit/nom_fit, color='red')
        bottom.set_ylim(.9,1.1)
        
        top.legend(title='Bin: {}'.format(i))
        top.set_xlabel('$m_{\ell\ell}$ [GeV]')
        top.set_ylabel('Counts')
        cms_format_fig(era, top, "\emph{Preliminary}")
        
        fig.savefig(f'{output_dir}/abcd_fit_variations/CR23_{era}_{i}.png')
    
    ratios_CR23[i] = {'up': up_fit/nom_fit, "down": down_fit/nom_fit}

In [ ]:
##
## create data hists
##
sp = get_stack_plotter(output_dir, era, bins='none')

In [ ]:
fig, ax = plt.subplots()
sp.binning = bins.bin_edges
sp.x_range = xrange

hist_SR1 = sp.draw_background(ax, 'DiLepMass', 'SR1')
hist_SR1_inv = hist_SR1.inverse_make_density_hist()
hist_SR1_inv.draw(ax)

data_SR1 = hist_SR1_inv.nominal

In [ ]:
fig, ax = plt.subplots()
sp.binning = bins.bin_edges
sp.x_range = (120, 401)

hist_SR2 = sp.draw_background(ax, 'DiLepMass', 'SR2')
hist_SR2_inv = hist_SR2.inverse_make_density_hist()
hist_SR2_inv.draw(ax)

data_SR2 = hist_SR2_inv.nominal

In [ ]:
##
## create workspace
##

In [ ]:
csvname = f'{output_dir}/combine_data/correlated/{era}/{era}_abcd_shapes_df_input.csv'
csvname

In [ ]:
def make_list(_list, channel, process, systematic, values, masses):
    for i, (x, m) in enumerate(zip(values, masses)):
        _list.append({'channel': channel, 'process': process,
                      'systematic': systematic, 'bin':m, 'sum_w': x, 'sum_ww':x})

In [ ]:
abcd['SR1']['bins']

In [ ]:
masses = Bins(abcd['SR1']['bins']).calc_bin_centers()

In [ ]:
csv_list = []

hist1 = abcd['SR1']['nom']
make_list(csv_list, 'SR1', 'ABCD', 'nominal', hist1, masses)

hist2 = abcd['SR2']['nom']
make_list(csv_list, 'SR2', 'ABCD', 'nominal', hist2, masses)

In [ ]:

for key, values in ratios_CR23.items():
    up, down = values['up'], values['down']
    make_list(csv_list, 'SR2', 'ABCD', f'fit_{key}_Up', up*hist2, masses)
    make_list(csv_list, 'SR2', 'ABCD', f'fit_{key}_Down', down*hist2, masses)
    
for key, values in ratios_CR13.items():
    up, down = values['up'], values['down']
    make_list(csv_list, 'SR1', 'ABCD', f'fit_{key}_Up', up*hist1, masses)
    make_list(csv_list, 'SR1', 'ABCD', f'fit_{key}_Down', down*hist1, masses)

In [ ]:
df = pd.DataFrame(csv_list)

df.to_csv(csvname, index=False)

In [ ]:
##
## envelope plots
##

In [ ]:
def array_from_df(_df, channel, process, systematic):
    _tdf = _df[(_df.channel==channel) & (_df.process==process) & (_df.systematic==systematic)]
    y = np.array(_tdf.sum_w)
    x = np.array(_tdf.bin)
    return x, y 

In [ ]:
#max, min
bins = df.bin.unique()
reg = 'SR1'
min_array = []
max_array = []

for mass in bins:
    tdf = df[(df.bin==mass) & (df.channel=='SR1')]
    min_bin = tdf.sum_w.min()
    max_bin = tdf.sum_w.max()
    min_array.append(min_bin)
    max_array.append(max_bin)
    
masses, nominal = array_from_df(df, reg, 'ABCD', 'nominal')

fig, axs = plt.subplots(2,1)

top, bottom = axs
top.plot(masses, max_array, label='Up', color='blue')

top.plot(masses, min_array, label='Down', color='red')

top.plot(masses, nominal, color='black')

bottom.plot(masses, max_array/nominal, color='blue')
bottom.plot(masses, nominal/nominal, color='black')
bottom.plot(masses, min_array/nominal, color='red')

##specific masses

def plot_var(ax, index, **kwargs):
    _, Up = array_from_df(df, reg, 'ABCD', f'fit_{index}_Up')
    _, Down = array_from_df(df, reg, 'ABCD', f'fit_{index}_Down')
    bottom.plot(masses, Up/nominal, color='red', **kwargs)
    bottom.plot(masses, Down/nominal, color='blue', **kwargs)
plot_var(bottom, 0, ls=':', alpha=.5)
plot_var(bottom, 110, ls='--', alpha=.5)
plot_var(bottom, 270, ls='-.', alpha=.5)

bottom.set_ylim(.95,1.05)

top.legend(title='Max/Min Envelope')
bottom.set_xlabel('$m_{\ell\ell}$ [GeV]')
top.set_ylabel('Counts')
bottom.set_ylabel('Ratio to Nominal')
cms_format_fig(era, top, "\emph{Preliminary}")

fig.savefig(f'{output_dir}/abcd_fit_variations/{reg}_min_max_envelope_{era}.png')

In [ ]:
#max, min
bins = df.bin.unique()
reg = 'SR2'
min_array = []
max_array = []

for mass in bins:
    tdf = df[(df.bin==mass) & (df.channel==reg)]
    min_bin = tdf.sum_w.min()
    max_bin = tdf.sum_w.max()
    min_array.append(min_bin)
    max_array.append(max_bin)
    
masses, nominal = array_from_df(df, reg, 'ABCD', 'nominal')

fig, axs = plt.subplots(2,1)

top, bottom = axs
top.plot(masses, max_array, label='Up', color='blue')

top.plot(masses, min_array, label='Down', color='red')

top.plot(masses, nominal, color='black')

bottom.plot(masses, max_array/nominal, color='blue')
bottom.plot(masses, nominal/nominal, color='black')
bottom.plot(masses, min_array/nominal, color='red')

##specific masses

def plot_var(ax, index, **kwargs):
    _, Up = array_from_df(df, reg, 'ABCD', f'fit_{index}_Up')
    _, Down = array_from_df(df, reg, 'ABCD', f'fit_{index}_Down')
    bottom.plot(masses, Up/nominal, color='red', **kwargs)
    bottom.plot(masses, Down/nominal, color='blue', **kwargs)
plot_var(bottom, 0, ls=':', alpha=.5)
plot_var(bottom, 110, ls='--', alpha=.5)
plot_var(bottom, 270, ls='-.', alpha=.5)

bottom.set_ylim(.95,1.05)

top.legend(title='Max/Min Envelope')
bottom.set_xlabel('$m_{\ell\ell}$ [GeV]')
top.set_ylabel('Counts')
bottom.set_ylabel('Ratio to Nominal')
cms_format_fig(era, top, "\emph{Preliminary}")

fig.savefig(f'{output_dir}/abcd_fit_variations/{reg}_min_max_envelope_{era}.png')

In [ ]:
import re
from mpl_toolkits.axes_grid1 import make_axes_locatable
def make_sys_matrix(reg):
    tdf = df[(df.systematic!='nominal') & (df.channel==reg)]
    masses = tdf['bin'].unique()
    up_matrix = []
    down_matrix = []
    nominal_matrix = []
    #noiminal 
    masses, nominal = array_from_df(df, reg, 'ABCD', 'nominal')

    
    for i, m in enumerate(masses):
        ttdf = tdf[(tdf.bin==m)]
        uttdf = ttdf[(ttdf.systematic.str.contains('Up'))]
        dttdf = ttdf[(ttdf.systematic.str.contains('Down'))]
        
        uValues = uttdf.sum_w
        u_sort = uttdf.systematic.apply(lambda x: int(re.findall('fit_([0-9]+)_', x)[0]))
        uValues = [x for _, x in sorted(zip(u_sort, uValues))]
        
        dValues = dttdf.sum_w
        d_sort = dttdf.systematic.apply(lambda x: int(re.findall('fit_([0-9]+)_', x)[0]))
        dValues = [x for _, x in sorted(zip(d_sort, dValues))]
        
        up_matrix.append(uValues)
        down_matrix.append(dValues)
        nominal_matrix.append([nominal[i] for _ in range(len(dValues))])
    return np.stack(up_matrix), np.stack(down_matrix), np.stack(nominal_matrix)
up, down, nominal = make_sys_matrix('SR1')

fig, ax = plt.subplots(2,1)
upim = ax[0].imshow(up/nominal, cmap = 'YlGn', interpolation = 'sinc', vmin = .98, vmax = 1.02)
divider = make_axes_locatable(ax[0])
cax = divider.append_axes('right', size='5%', pad=0.05)
fig.colorbar(upim, cax=cax, orientation='vertical')
ax[0].set_xlabel('j')
ax[0].set_ylabel('i')


downim = ax[1].imshow(down/nominal, cmap = 'YlGn', interpolation = 'sinc', vmin = .98, vmax = 1.02)
divider = make_axes_locatable(ax[1])
cax = divider.append_axes('right', size='5%', pad=0.05)
fig.colorbar(downim, cax=cax, orientation='vertical')
ax[1].set_xlabel('j')
ax[1].set_ylabel('i')

cms_format_fig(era, ax[0], '\emph{Preliminary}')

fig.savefig(f'{output_dir}/abcd_fit_variations/Matrix_SR1_{era}.png')

In [ ]:
import re
from mpl_toolkits.axes_grid1 import make_axes_locatable
def make_sys_matrix(reg):
    tdf = df[(df.systematic!='nominal') & (df.channel==reg)]
    masses = tdf['bin'].unique()
    up_matrix = []
    down_matrix = []
    nominal_matrix = []
    #noiminal 
    masses, nominal = array_from_df(df, reg, 'ABCD', 'nominal')

    
    for i, m in enumerate(masses):
        ttdf = tdf[(tdf.bin==m)]
        uttdf = ttdf[(ttdf.systematic.str.contains('Up'))]
        dttdf = ttdf[(ttdf.systematic.str.contains('Down'))]
        
        uValues = uttdf.sum_w
        u_sort = uttdf.systematic.apply(lambda x: int(re.findall('fit_([0-9]+)_', x)[0]))
        uValues = [x for _, x in sorted(zip(u_sort, uValues))]
        
        dValues = dttdf.sum_w
        d_sort = dttdf.systematic.apply(lambda x: int(re.findall('fit_([0-9]+)_', x)[0]))
        dValues = [x for _, x in sorted(zip(d_sort, dValues))]
        
        up_matrix.append(uValues)
        down_matrix.append(dValues)
        nominal_matrix.append([nominal[i] for _ in range(len(dValues))])
    return np.stack(up_matrix), np.stack(down_matrix), np.stack(nominal_matrix)
up, down, nominal = make_sys_matrix('SR2')

fig, ax = plt.subplots(2,1)
upim = ax[0].imshow(up/nominal, cmap = 'YlGn', interpolation = 'sinc', vmin = .98, vmax = 1.02)
divider = make_axes_locatable(ax[0])
cax = divider.append_axes('right', size='5%', pad=0.05)
fig.colorbar(upim, cax=cax, orientation='vertical')
ax[0].set_xlabel('j')
ax[0].set_ylabel('i')


downim = ax[1].imshow(down/nominal, cmap = 'YlGn', interpolation = 'sinc', vmin = .98, vmax = 1.02)
divider = make_axes_locatable(ax[1])
cax = divider.append_axes('right', size='5%', pad=0.05)
fig.colorbar(downim, cax=cax, orientation='vertical')
ax[1].set_xlabel('j')
ax[1].set_ylabel('i')



cms_format_fig(era, ax[0], '\emph{Preliminary}')
fig.savefig(f'{output_dir}/abcd_fit_variations/Matrix_SR2_{era}.png')

In [ ]:
fig, ax = plt.subplots(2,1)

up, down, nominal = make_sys_matrix('SR1')
upKeys = up> down
ax[0].imshow(upKeys)

up, down, nominal = make_sys_matrix('SR2')
upKeys = up> down
ax[1].imshow(upKeys, interpolation = 'sinc')

cms_format_fig(era, ax[0], '\emph{Preliminary}')
fig.savefig(f'{output_dir}/abcd_fit_variations/Up_is_bigger_{era}.png')

In [ ]:
# quadrature


#max, min
bins = df.bin.unique()
reg = 'SR1'
min_array = []
max_array = []

up, down, nominal = make_sys_matrix(reg)
up = up-nominal
down = down-nominal

masses, nominal = array_from_df(df, reg, 'ABCD', 'nominal')

for i, (up_bin, down_bin) in enumerate(zip(up, down)):
    up_sum  = ((up_bin[up_bin>0]**2).sum() + (down_bin[down_bin>0]**2).sum())**.5 + nominal[i]
    down_sum  = -((up_bin[up_bin<0]**2).sum() + (down_bin[down_bin<0]**2).sum())**.5 + nominal[i]
    min_array.append(down_sum)
    max_array.append(up_sum)
    
min_array = np.array(min_array)
max_array = np.array(max_array)   


fig, axs = plt.subplots(2,1)

top, bottom = axs
top.plot(masses, max_array, label='Up', color='blue')

top.plot(masses, min_array, label='Down', color='red')

top.plot(masses, nominal, color='black')

bottom.plot(masses, max_array/nominal, color='blue')
bottom.plot(masses, nominal/nominal, color='black')
bottom.plot(masses, min_array/nominal, color='red')

##specific masses

def plot_var(ax, index, **kwargs):
    _, Up = array_from_df(df, reg, 'ABCD', f'fit_{index}_Up')
    _, Down = array_from_df(df, reg, 'ABCD', f'fit_{index}_Down')
    bottom.plot(masses, Up/nominal, color='red', **kwargs)
    bottom.plot(masses, Down/nominal, color='blue', **kwargs)
plot_var(bottom, 0, ls=':', alpha=.5)
plot_var(bottom, 110, ls='--', alpha=.5)
plot_var(bottom, 270, ls='-.', alpha=.5)

bottom.set_ylim(.80,1.20)

top.legend(title='Quadrature')
bottom.set_xlabel('$m_{\ell\ell}$ [GeV]')
top.set_ylabel('Counts')
bottom.set_ylabel('Ratio to Nominal')
cms_format_fig(era, top, "\emph{Preliminary}")

fig.savefig(f'{output_dir}/abcd_fit_variations/{reg}_quadrature.png')

In [ ]:
# quadrature


#max, min
bins = df.bin.unique()
reg = 'SR2'
min_array = []
max_array = []

up, down, nominal = make_sys_matrix(reg)
up = up-nominal
down = down-nominal

masses, nominal = array_from_df(df, reg, 'ABCD', 'nominal')

for i, (up_bin, down_bin) in enumerate(zip(up, down)):
    up_sum  = ((up_bin[up_bin>0]**2).sum() + (down_bin[down_bin>0]**2).sum())**.5 + nominal[i]
    down_sum  = -((up_bin[up_bin<0]**2).sum() + (down_bin[down_bin<0]**2).sum())**.5 + nominal[i]
    min_array.append(down_sum)
    max_array.append(up_sum)
    
min_array = np.array(min_array)
max_array = np.array(max_array)   


fig, axs = plt.subplots(2,1)

top, bottom = axs
top.plot(masses, max_array, label='Up', color='blue')

top.plot(masses, min_array, label='Down', color='red')

top.plot(masses, nominal, color='black')

bottom.plot(masses, max_array/nominal, color='blue')
bottom.plot(masses, nominal/nominal, color='black')
bottom.plot(masses, min_array/nominal, color='red')

##specific masses

def plot_var(ax, index, **kwargs):
    _, Up = array_from_df(df, reg, 'ABCD', f'fit_{index}_Up')
    _, Down = array_from_df(df, reg, 'ABCD', f'fit_{index}_Down')
    bottom.plot(masses, Up/nominal, color='red', **kwargs)
    bottom.plot(masses, Down/nominal, color='blue', **kwargs)
plot_var(bottom, 0, ls=':', alpha=.5)
plot_var(bottom, 110, ls='--', alpha=.5)
plot_var(bottom, 270, ls='-.', alpha=.5)

bottom.set_ylim(.80,1.20)

top.legend(title='Quadrature')
bottom.set_xlabel('$m_{\ell\ell}$ [GeV]')
top.set_ylabel('Counts')
bottom.set_ylabel('Ratio to Nominal')
cms_format_fig(era, top, "\emph{Preliminary}")

fig.savefig(f'{output_dir}/abcd_fit_variations/{reg}_quadrature.png')

In [ ]:
##
## signal inection
##

In [ ]:
#make signal injection df

signal_csv  = f'{output_dir}/combine_data/{era}/{era}_signal_shapes_df_input.csv'
signal_df = pd.read_csv(signal_csv)


In [ ]:
def array_from_df(_df, channel, process, systematic):
    _tdf = _df[(_df.channel==channel) & (_df.process==process) & (_df.systematic==systematic)]
    y = np.array(_tdf.sum_w)
    x = np.array(_tdf.bin)
    return x, y 

In [ ]:
for x in signal_df.process.unique():
    csv_list = []
    for reg in ['SR1', 'SR2']:
        _, signal  = array_from_df(signal_df, reg, x, 'nominal')
        if reg=='SR1': make_list(csv_list, reg, 'data_obs', 'nominal', signal+hist1, masses)
        if reg=='SR2': make_list(csv_list, reg, 'data_obs', 'nominal', signal+hist2, masses)
    df = pd.DataFrame(csv_list)
    csvname = f'{output_dir}/combine_data/correlated/{era}/{era}_{x}_abcd_shapes_df_input.csv'
    df.to_csv(csvname, index=False)

In [ ]:
csv_list = []
make_list(csv_list, 'SR2', 'data_obs', 'nominal', hist2, masses)
make_list(csv_list, 'SR1', 'data_obs', 'nominal', hist1, masses)
df = pd.DataFrame(csv_list)
x = 'nominal'
csvname = f'{output_dir}/combine_data/correlated/{era}/{era}_{x}_abcd_shapes_df_input.csv'
df.to_csv(csvname, index=False)

In [ ]:
csvname